# Alocação dinâmica de vetores e matrizes — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial retoma o `notas.c` visto em aula: a turma cujo número de alunos e de avaliações
só é conhecido quando o programa roda. Depois de quatro blocos de prática — o vetor, as duas
formas de matriz e os erros — vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Explicar por que um ponteiro precisa andar acompanhado do tamanho do bloco;
- Alocar e liberar uma matriz na forma `double **`, na ordem certa, tratando falha no meio do laço;
- Alocar e liberar a mesma matriz como um bloco único, indexando com `i * nc + j`;
- Justificar a escolha entre as duas formas em termos de contiguidade, custo e sintaxe;
- Reconhecer os três erros clássicos de matriz dinâmica e usar o sanitizer para localizá-los.

In [ ]:
# Confira se o gcc está disponível no seu ambiente
!gcc --version | head -1

## O programa da aula

Recompile e execute o programa condutor **exatamente como saiu da aula** e confira que a
saída bate com a que vimos no slide.

Os endereços da linha `B:` mudam a cada execução e são diferentes na sua máquina — o que
interessa é a **diferença** entre eles, comparada com o tamanho de uma linha (linha `C:`).

In [ ]:
%%writefile notas.c
/* notas.c --- medias de uma turma cujo tamanho so se sabe rodando */
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>

double **cria_matriz(int nl, int nc)
{
    double **m = malloc(nl * sizeof(double *));
    if (m == NULL)
        return NULL;
    for (int i = 0; i < nl; i++) {
        m[i] = malloc(nc * sizeof(double));
        if (m[i] == NULL) {                 /* falhou no meio do caminho */
            for (int k = 0; k < i; k++)
                free(m[k]);
            free(m);
            return NULL;
        }
    }
    return m;
}

void libera_matriz(double **m, int nl)
{
    for (int i = 0; i < nl; i++)
        free(m[i]);
    free(m);
}

void medias_por_aluno(double **notas, int nl, int nc, double *saida)
{
    for (int i = 0; i < nl; i++) {
        double soma = 0.0;
        for (int j = 0; j < nc; j++)
            soma += notas[i][j];
        saida[i] = soma / nc;
    }
}

int main(void)
{
    int nalunos = 3, navaliacoes = 4;       /* viriam de um scanf */
    double valores[] = { 8.0, 7.5,  6.0, 9.0,
                         5.5, 6.0,  7.0, 4.5,
                         9.5, 10.0, 8.5, 9.0 };

    double **notas  = cria_matriz(nalunos, navaliacoes);
    double  *medias = malloc(nalunos * sizeof(double));
    if (notas == NULL || medias == NULL) {
        fprintf(stderr, "memoria insuficiente\n");
        return 1;
    }

    for (int i = 0; i < nalunos; i++)
        for (int j = 0; j < navaliacoes; j++)
            notas[i][j] = valores[i * navaliacoes + j];

    medias_por_aluno(notas, nalunos, navaliacoes, medias);
    for (int i = 0; i < nalunos; i++)
        printf("A: aluno %d -> media %.2f\n", i, medias[i]);

    for (int i = 0; i < nalunos; i++) {
        printf("B: linha %d comeca em %p", i, (void *) notas[i]);
        if (i > 0)
            printf("  (%ld bytes depois da linha %d)",
                   (long) ((uintptr_t) notas[i] - (uintptr_t) notas[i - 1]), i - 1);
        printf("\n");
    }
    printf("C: uma linha ocupa %zu bytes\n", navaliacoes * sizeof(double));
    printf("D: sizeof(notas) = %zu\n", sizeof(notas));

    free(medias);
    libera_matriz(notas, nalunos);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra notas.c -o notas && ./notas
!./notas | grep -q 'A: aluno 0 -> media 7.62' && echo OK \
  || echo 'Verifique: esperava media 7.62 para o aluno 0'

## 1. Um vetor dinâmico é um ponteiro *mais* um número

Um vetor declarado (`double fixo[3]`) e um vetor alocado (`malloc(3 * sizeof(double))`) são
usados do mesmo jeito: `v[i]`. Mas só o primeiro carrega o próprio tamanho.

`sizeof` de um vetor declarado devolve os bytes do vetor; `sizeof` de um ponteiro devolve os
bytes do ponteiro — 8, sempre, não importa o tamanho do bloco. Por isso toda função que recebe
um vetor dinâmico recebe também quantos elementos ele tem.

In [ ]:
%%writefile vetor.c
/* vetor.c --- o ponteiro sabe onde comeca, nao sabe onde termina */
#include <stdio.h>
#include <stdlib.h>

void mostra(const double *v, int n)     /* n precisa vir junto */
{
    for (int i = 0; i < n; i++)
        printf("%.2f ", v[i]);
    printf("\n");
}

int main(void)
{
    int n = 3;
    double fixo[3] = { 1.0, 2.0, 3.0 };
    double *dinamico = malloc(n * sizeof(double));
    if (dinamico == NULL)
        return 1;
    for (int i = 0; i < n; i++)
        dinamico[i] = (i + 1) * 10.0;

    printf("sizeof(fixo)     = %zu  -> da para descobrir n: %zu\n",
           sizeof(fixo), sizeof(fixo) / sizeof(fixo[0]));
    printf("sizeof(dinamico) = %zu  -> e o tamanho do ponteiro\n",
           sizeof(dinamico));

    mostra(fixo, n);
    mostra(dinamico, n);

    free(dinamico);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra vetor.c -o vetor && ./vetor

## 2. A matriz como vetor de ponteiros

`double **m` é um bloco de **ponteiros**; cada `m[i]` aponta para um bloco separado — a linha
`i`. São `1 + nl` chamadas a `malloc` e `1 + nl` chamadas a `free`.

Duas consequências aparecem nos experimentos abaixo:

1. as linhas **não** precisam ficar vizinhas na memória;
2. como cada linha tem seu próprio `malloc`, cada uma pode ter seu próprio tamanho.

No `notas.c` as três linhas saíram encostadas (32 bytes de distância, exatamente o tamanho de
uma linha). O programa a seguir mostra que isso é circunstância, não garantia: basta alguém
pedir memória no meio do laço.

In [ ]:
%%writefile espalha.c
/* espalha.c --- as linhas ficam encostadas? depende de quem mais pediu memoria */
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>

int main(void)
{
    int nl = 4, nc = 4;
    double **m = malloc(nl * sizeof(double *));
    char *intruso[4];

    for (int i = 0; i < nl; i++) {
        m[i] = malloc(nc * sizeof(double));   /* uma linha... */
        intruso[i] = malloc(nc * sizeof(double));  /* ...e outro pedido do mesmo tamanho */
    }

    printf("uma linha ocupa %zu bytes\n", nc * sizeof(double));
    for (int i = 0; i < nl; i++) {
        printf("linha %d em %p", i, (void *) m[i]);
        if (i > 0)
            printf("  (%ld bytes depois da anterior)",
                   (long) ((uintptr_t) m[i] - (uintptr_t) m[i - 1]));
        printf("\n");
    }

    for (int i = 0; i < nl; i++) { free(m[i]); free(intruso[i]); }
    free(m);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra espalha.c -o espalha && ./espalha

Compare as distâncias com as do `notas.c`. Elas dobraram — cada pedido do `intruso` se
intercalou entre duas linhas.

> Rode a célula acima algumas vezes. Os endereços mudam; a distância, não. Agora **comente** a
> linha do `intruso` e rode de novo: as linhas voltam a ficar encostadas. Quem decide é o
> alocador.

O outro lado da moeda é a matriz irregular — cada aluno com um número diferente de avaliações.
Repare que o vetor `quantas` passa a ser obrigatório: sem ele, ninguém sabe onde cada linha
termina.

In [ ]:
%%writefile irregular.c
/* irregular.c --- cada aluno com um numero diferente de avaliacoes */
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int nl = 3;
    int quantas[] = { 2, 4, 3 };                  /* avaliacoes de cada aluno */
    double **notas = malloc(nl * sizeof(double *));

    for (int i = 0; i < nl; i++) {
        notas[i] = malloc(quantas[i] * sizeof(double));
        for (int j = 0; j < quantas[i]; j++)
            notas[i][j] = 5.0 + i + j;
    }

    for (int i = 0; i < nl; i++) {
        double soma = 0.0;
        for (int j = 0; j < quantas[i]; j++)
            soma += notas[i][j];
        printf("aluno %d: %d notas, media %.2f\n", i, quantas[i], soma / quantas[i]);
    }

    for (int i = 0; i < nl; i++) free(notas[i]);
    free(notas);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra irregular.c -o irregular && ./irregular

## 3. A matriz como um bloco único

Um `malloc` só, com as `nl * nc` casas em sequência. A casa `(i, j)` fica no índice
`i * nc + j`: pular `i` linhas inteiras custa `i * nc` casas, e dentro da linha andam-se mais
`j`.

Duas coisas para reparar no código:

- o `(size_t)` no primeiro fator do produto: sem ele, `nl * nc` é calculado em `int` e estoura
  em matrizes grandes (`50000 * 50000` não cabe em `int`);
- `nc` aparece na assinatura de toda função que acessa a matriz, porque sem ele não há como
  calcular o índice.

In [ ]:
%%writefile contigua.c
/* contigua.c --- a mesma matriz, agora em um unico bloco */
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>

#define EM(m, i, j, nc)  ((m)[(i) * (nc) + (j)])

void medias_por_aluno(const double *notas, int nl, int nc, double *saida)
{
    for (int i = 0; i < nl; i++) {
        double soma = 0.0;
        for (int j = 0; j < nc; j++)
            soma += notas[i * nc + j];
        saida[i] = soma / nc;
    }
}

int main(void)
{
    int nl = 3, nc = 4;
    double valores[] = { 8.0, 7.5,  6.0, 9.0,
                         5.5, 6.0,  7.0, 4.5,
                         9.5, 10.0, 8.5, 9.0 };

    double *notas = malloc((size_t) nl * nc * sizeof(double));  /* um malloc so */
    double *medias = malloc((size_t) nl * sizeof(double));
    if (notas == NULL || medias == NULL) {
        fprintf(stderr, "memoria insuficiente\n");
        return 1;
    }

    for (int i = 0; i < nl; i++)
        for (int j = 0; j < nc; j++)
            EM(notas, i, j, nc) = valores[i * nc + j];

    medias_por_aluno(notas, nl, nc, medias);
    for (int i = 0; i < nl; i++)
        printf("aluno %d: media %.2f\n", i, medias[i]);

    for (int i = 0; i < nl; i++)
        printf("linha %d comeca em %p  (%ld bytes depois do inicio)\n",
               i, (void *) (notas + i * nc),
               (long) ((uintptr_t) (notas + i * nc) - (uintptr_t) notas));

    free(medias);
    free(notas);          /* um free so */
    return 0;
}

In [ ]:
!gcc -Wall -Wextra contigua.c -o contigua && ./contigua

Agora as distâncias são exatas: 0, 32, 64. E continuarão iguais em qualquer execução, em
qualquer máquina — a contiguidade veio da forma de alocar, não da sorte.

### O índice trocado

Este é o erro mais perigoso da aula porque **nenhuma ferramenta o acusa**. Para quaisquer `i` e
`j` válidos, `j * nl + i` também cai dentro do bloco: o maior valor possível continua sendo
`nl * nc - 1`. Não há acesso inválido — só números no lugar errado.

In [ ]:
%%writefile troca.c
/* troca.c --- o indice trocado: i*nc+j virou j*nl+i */
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int nl = 3, nc = 4;
    double *m = malloc((size_t) nl * nc * sizeof(double));

    for (int i = 0; i < nl; i++)
        for (int j = 0; j < nc; j++)
            m[i * nc + j] = 10 * i + j;        /* guarda certo */

    for (int i = 0; i < nl; i++) {
        for (int j = 0; j < nc; j++)
            printf("%5.0f", m[j * nl + i]);    /* le errado */
        printf("\n");
    }

    free(m);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra troca.c -o troca && ./troca
# esperado:  0  1  2  3 / 10 11 12 13 / 20 21 22 23 -- o que saiu e a transposta

## 4. Os erros que o compilador não pega

As três células a seguir **devem falhar**. Compile cada uma com `-fsanitize=address` e leia a
mensagem inteira: ela diz o que aconteceu, em que linha, e — quando é o caso — em que linha
aquela memória tinha sido liberada.

### Erro 1: liberar de fora para dentro

In [ ]:
%%writefile ordem.c
/* ordem.c --- liberar na ordem errada */
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int nl = 3, nc = 4;
    double **m = malloc(nl * sizeof(double *));
    for (int i = 0; i < nl; i++)
        m[i] = malloc(nc * sizeof(double));

    free(m);                       /* o vetor de ponteiros se foi... */
    for (int i = 0; i < nl; i++)
        free(m[i]);                /* ...e agora m[i] le memoria liberada */

    printf("cheguei ao fim\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -g -fsanitize=address ordem.c -o ordem && ./ordem

Os endereços das linhas moram *dentro* do bloco de ponteiros. Liberar `m` primeiro é o mesmo
que jogar fora a lista de endereços antes de usá-la.

### Erro 2: o ponteiro que andou

In [ ]:
%%writefile perdeu.c
/* perdeu.c --- o ponteiro andou e o comeco do bloco se perdeu */
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int n = 4;
    double *v = malloc(n * sizeof(double));

    for (int i = 0; i < n; i++) {
        *v = i;          /* escreve na posicao atual... */
        v++;             /* ...e anda uma casa */
    }

    free(v);             /* v nao aponta mais para o inicio do bloco */
    printf("cheguei ao fim\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -g -fsanitize=address perdeu.c -o perdeu && ./perdeu

`free` exige exatamente o endereço que o `malloc` devolveu. Se precisar caminhar pelo bloco,
caminhe com uma cópia.

### Erro 3: um passo além do fim da linha

In [ ]:
%%writefile estouro.c
/* estouro.c --- um passo alem do fim da linha */
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int nl = 3, nc = 4;
    double **m = malloc(nl * sizeof(double *));
    for (int i = 0; i < nl; i++)
        m[i] = malloc(nc * sizeof(double));

    for (int i = 0; i < nl; i++)
        for (int j = 0; j <= nc; j++)      /* <= em vez de < */
            m[i][j] = 0.0;

    printf("cheguei ao fim\n");
    for (int i = 0; i < nl; i++) free(m[i]);
    free(m);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -g -fsanitize=address estouro.c -o estouro && ./estouro

## Tarefas

Trabalhe sobre uma cópia do programa condutor.

**Tarefa 1.** Acrescente `medias_por_avaliacao` — a média da coluna `j`, sobre todos os alunos.
Repare em qual dos dois laços fica por fora e explique por que essa versão lê a memória
"pulando".

**Tarefa 2.** Faça `nalunos` e `navaliacoes` virem de um `scanf` e as notas virem do teclado
(no Colab, use uma string de entrada ou mantenha os valores fixos e apenas troque as
dimensões). Teste com 1 aluno e 1 avaliação, e com 0 alunos: o programa sobrevive aos dois?

**Tarefa 3.** Reescreva o `notas.c` inteiro na forma de bloco único. Compare as duas versões:
quantas linhas mudaram? Quantos `free` sobraram?

In [ ]:
!cp notas.c notas_tarefa.c
# edite notas_tarefa.c e rode esta célula de novo
!gcc -Wall -Wextra notas_tarefa.c -o notas_tarefa && ./notas_tarefa

## Desafio: a turma que muda de tamanho

Escreva `adicionar_aluno(double ***notas, int *nl, int nc)`, que acrescenta uma linha zerada à
matriz, e `remover_aluno(double ***notas, int *nl, int quem)`, que apaga uma linha.

Requisitos:

- use `realloc` no bloco de ponteiros com o padrão do ponteiro temporário — se ele falhar, a
  matriz antiga deve continuar utilizável;
- aloque a linha nova com `calloc`, e desfaça o `realloc` se ela falhar;
- ao remover, libere a linha **antes** de perder o endereço dela, e feche o buraco deslocando
  as linhas seguintes;
- remover uma linha que não existe deve deixar tudo como está;
- o programa deve terminar sem vazamentos e sem nenhum erro do sanitizer.

Por que `double ***`? Porque o `realloc` pode mudar o endereço do bloco de ponteiros, e quem
chamou a função precisa ficar sabendo do novo endereço.

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <stdlib.h>

/* TODO: acrescenta uma linha zerada no fim da matriz e atualiza *nl.
   Devolve 1 em caso de sucesso e 0 se faltar memoria -- e, se faltar,
   a matriz antiga precisa continuar utilizavel.
   Dica: 'notas' e double *** porque o realloc pode mudar o endereco do
   bloco de ponteiros, e quem chamou precisa ficar sabendo. */
int adicionar_aluno(double ***notas, int *nl, int nc)
{
    (void) notas; (void) nl; (void) nc;
    /* TODO: implemente aqui */
    return 0;
}

/* TODO: remove a linha 'quem', libera a memoria dela, fecha o buraco
   deslocando as linhas seguintes e atualiza *nl. */
int remover_aluno(double ***notas, int *nl, int quem)
{
    (void) notas; (void) nl; (void) quem;
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    int nl = 2, nc = 3;
    double **notas = malloc(nl * sizeof(double *));
    for (int i = 0; i < nl; i++) {
        notas[i] = malloc(nc * sizeof(double));
        for (int j = 0; j < nc; j++)
            notas[i][j] = 10 * i + j;
    }

    /* TODO: acrescente um aluno, remova o aluno 0, imprima a matriz
       e libere tudo antes de sair. */
    printf("antes: %d alunos\n", nl);

    for (int i = 0; i < nl; i++)
        free(notas[i]);
    free(notas);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -g -fsanitize=address desafio.c -o desafio && ./desafio
# No Linux (Colab), o proprio sanitizer acusa vazamentos ao terminar.
# No macOS, use:  MallocStackLogging=1 leaks -atExit -- ./desafio | grep 'leaks for'

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Para esta aula, os pontos de partida
são o capítulo 5.6–5.9 de Kernighan & Ritchie (ponteiros, vetores de ponteiros e arranjos
multidimensionais) e o capítulo 12 de Backes.